# 📊 Gerador de Dataset de Gestos (Refatorado)

Este notebook foi refatorado para ser mais modular e eficiente. Ele captura landmarks das mãos e os salva em um CSV estruturado.

### ✨ Melhorias:
- **Arquitetura modular**: Funções dedicadas para desenho, processamento e gravação.
- **Gerenciamento Automático**: O arquivo CSV e as ferramentas do MediaPipe são inicializados de forma mais limpa.
- **UX Premium**: Informações na tela mais organizadas.

In [1]:
import sys
import os

# Força o notebook a me dizer qual Python ele está usando
print(f"🐍 Python sendo usado: {sys.executable}")
print(f"📂 Pasta atual: {os.getcwd()}")

try:
    import cv2
    import mediapipe as mp
    import numpy as np
    print("✅ SUCESSO: OpenCV, MediaPipe e Numpy carregados!")
except ImportError as e:
    print(f"❌ ERRO DE IMPORTAÇÃO: {e}")
    print("\n💡 Dica: Verifique se você selecionou o kernel 'Python (Gestos)' no topo direito do VS Code.")


🐍 Python sendo usado: c:\Users\ednar\OneDrive\Imagens\recog_system\activate\Scripts\python.exe
📂 Pasta atual: c:\Users\ednar\OneDrive\Imagens\recog_system
✅ SUCESSO: OpenCV, MediaPipe e Numpy carregados!


## ⚙️ Configurações
Ajuste o rótulo antes de iniciar a captura de cada gesto.

In [2]:
CONFIG = {
    "csv_file": "gesture_dataset.csv",
    "target_label": "V_SIGN",  # Altere para o gesto que vai capturar
    "model_path": "gesture_recognizer.task",
    "num_hands": 2
}

In [3]:
def initialize_csv(filename):
    """Cria o arquivo CSV com cabeçalho se não existir."""
    if not os.path.exists(filename):
        header = ["target"]
        for i in range(21):
            header += [f"x{i}", f"y{i}", f"z{i}"]
        
        with open(filename, 'w', newline='') as f:
            csv.writer(f).writerow(header)
        return f"Novo arquivo '{filename}' criado."
    return f"Anexando dados ao arquivo '{filename}'."

print(initialize_csv(CONFIG["csv_file"]))

Anexando dados ao arquivo 'gesture_dataset.csv'.


In [4]:
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision
import os

def load_recognizer(model_path, num_hands):
    """Carrega o MediaPipe de forma independente."""
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"❌ Modelo não encontrado: {model_path}")

    # Usamos mp_python para evitar conflito com o nome 'python'
    base_options = mp_python.BaseOptions(model_asset_path=model_path)
    options = vision.GestureRecognizerOptions(
        base_options=base_options, 
        num_hands=num_hands
    )
    
    print(f"🔄 Carregando modelo '{model_path}'...")
    model = vision.GestureRecognizer.create_from_options(options)
    print("✅ Sucesso! MediaPipe pronto para uso.")
    return model

# Garante que as configurações existam nesta célula
if 'CONFIG' not in locals():
    CONFIG = {
        "csv_file": "gesture_dataset.csv",
        "target_label": "V_SIGN",
        "model_path": "gesture_recognizer.task",
        "num_hands": 2
    }

# Executa o carregamento
recognizer = load_recognizer(CONFIG["model_path"], CONFIG["num_hands"])


🔄 Carregando modelo 'gesture_recognizer.task'...
✅ Sucesso! MediaPipe pronto para uso.


In [5]:
def draw_ui(img, label, count):
    """Desenha a interface de usuário no frame."""
    # Painel de fundo
    cv2.rectangle(img, (10, 10), (300, 90), (0, 0, 0), -1)
    cv2.rectangle(img, (10, 10), (300, 90), (0, 255, 0), 1)
    
    cv2.putText(img, f"Gesto: {label}", (25, 45), 
                cv2.FONT_HERSHEY_DUPLEX, 0.7, (255, 255, 255), 1)
    cv2.putText(img, f"Capturas: {count}", (25, 75), 
                cv2.FONT_HERSHEY_DUPLEX, 0.7, (0, 255, 0), 1)
    
    cv2.putText(img, "[K] Gravar  [Q] Sair", (10, img.shape[0] - 20), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)

def draw_skeleton(img, landmarks):
    """Desenha as articulações e conexões da mão."""
    h, w, _ = img.shape
    pts = [(int(lm.x * w), int(lm.y * h)) for lm in landmarks]
    
    # Conexões simplificadas para o recorder
    connections = [
        (0,1,2,3,4), (0,5,6,7,8), (5,9,10,11,12), 
        (9,13,14,15,16), (13,17,18,19,20), (0,17)
    ]
    
    for group in connections:
        for i in range(len(group)-1):
            cv2.line(img, pts[group[i]], pts[group[i+1]], (0, 255, 0), 2)
            
    for pt in pts:
        cv2.circle(img, pt, 4, (255, 255, 255), -1)
        cv2.circle(img, pt, 2, (0, 0, 0), -1)

In [6]:
import cv2
import mediapipe as mp
import csv
import os

# --- Verificação de Segurança ---
if 'recognizer' not in locals():
    print("❌ ERRO: O 'recognizer' não foi carregado. Rode a célula anterior primeiro!")
else:
    try:
        cap = cv2.VideoCapture(0)
        session_counter = 0
        print("✅ Sistema iniciado! Foque na janela da câmera.")
        print("⌨️ Comandos: [K] Capturar Gesto  |  [Q] Sair")

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            frame = cv2.flip(frame, 1)
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
            
            result = recognizer.recognize(mp_image)
            
            # Desenha os landmarks (se as funções draw_ existirem)
            if result.hand_landmarks:
                for hand in result.hand_landmarks:
                    if 'draw_skeleton' in locals():
                        draw_skeleton(frame, hand)
            
            if 'draw_ui' in locals():
                draw_ui(frame, CONFIG["target_label"], session_counter)
            
            cv2.imshow('Gesture Dataset Recorder', frame)
            
            key = cv2.waitKey(1) & 0xFF
            
            # Atalho K para salvar no CSV
            if key == ord('k') and result.hand_landmarks:
                # Salva apenas a primeira mão detectada
                row = [CONFIG["target_label"]] + [v for lm in result.hand_landmarks[0] for v in (lm.x, lm.y, lm.z)]
                
                with open(CONFIG["csv_file"], 'a', newline='') as f:
                    csv.writer(f).writerow(row)
                    
                session_counter += 1
                print(f"📊 Capturas salvas: {session_counter}", end="\r")

            elif key == ord('q'):
                break

    finally:
        if 'cap' in locals() and cap.isOpened():
            cap.release()
        cv2.destroyAllWindows()
        print(f"\n🏁 Sessão finalizada. Total capturado: {session_counter}")


✅ Sistema iniciado! Foque na janela da câmera.
⌨️ Comandos: [K] Capturar Gesto  |  [Q] Sair
📊 Capturas salvas: 113
🏁 Sessão finalizada. Total capturado: 113
